In [ ]:
def split_geotiff(input_path, output_dir, tile_width=256, tile_height=256, skip_partial_tiles=False):
	"""
	Split a GeoTIFF file into smaller tiles.
	
	Parameters:
	-----------
	input_path : str
		Path to the input GeoTIFF file
	output_dir : str
		Directory to save output tiles
	tile_width : int
		Width of each tile in pixels (default: 256)
	tile_height : int
		Height of each tile in pixels (default: 256)
	skip_partial_tiles : bool
		If True, skip tiles at the edges that would be smaller than tile_width x tile_height
		If False, include partial tiles at edges (default: False)
	
	Returns:
	--------
	list : List of output tile file paths
	"""
	import os
	import numpy as np
	import rasterio
	from rasterio.windows import Window
	from pathlib import Path
	
	# Error handling for invalid inputs
	if not input_path:
		raise ValueError("Input path cannot be empty or None")
	
	if not os.path.exists(input_path):
		raise FileNotFoundError(f"Input file not found: {input_path}")
	
	if tile_width <= 0 or tile_height <= 0:
		raise ValueError("Tile dimensions must be positive integers")
	
	# Create output directory if it doesn't exist
	os.makedirs(output_dir, exist_ok=True)
	
	try:
		# Open the input GeoTIFF
		with rasterio.open(input_path) as src:
			# Get image properties
			width = src.width
			height = src.height
			bands = src.count
			crs = src.crs
			transform = src.transform
			driver = src.driver
			dtype = src.dtypes[0]
			nodata = src.nodata
			
			print(f"Input GeoTIFF: {input_path}")
			print(f"Dimensions: {width} x {height} pixels, {bands} band(s)")
			print(f"CRS: {crs}")
			print(f"Tile size: {tile_width} x {tile_height}")
			
			# Calculate number of tiles in each direction
			n_cols = int(np.ceil(width / tile_width))
			n_rows = int(np.ceil(height / tile_height))
			
			print(f"Will create {n_cols} x {n_rows} = {n_cols * n_rows} tiles")
			
			output_files = []
			
			# Iterate through rows and columns
			for row in range(n_rows):
				for col in range(n_cols):
					# Calculate window boundaries
					col_off = col * tile_width
					row_off = row * tile_height
					
					# Calculate actual tile size (may be smaller at edges)
					win_width = min(tile_width, width - col_off)
					win_height = min(tile_height, height - row_off)
					
					# Skip partial tiles if requested
					if skip_partial_tiles:
						if win_width < tile_width or win_height < tile_height:
							print(f"Skipping partial tile at row={row}, col={col} ({win_width}x{win_height})")
							continue
					
					# Create window for this tile
					window = Window(col_off, row_off, win_width, win_height)
					
					# Read data for this window
					data = src.read(window=window)
					
					# Calculate the geotransform for this tile
					# The transform describes how to convert pixel coordinates to geographic coordinates
					tile_transform = rasterio.windows.transform(window, transform)
					
					# Generate output filename with position info
					# Get the top-left corner coordinates for the filename
					x_min, y_max = tile_transform * (0, 0)
					x_min_int = int(x_min)
					y_max_int = int(y_max)
					
					output_filename = f"tile_{x_min_int}_{y_max_int}_{win_width}_{win_height}.tif"
					output_path = os.path.join(output_dir, output_filename)
					
					# Create output metadata
					output_meta = {
						'driver': driver,
						'height': win_height,
						'width': win_width,
						'count': bands,
						'dtype': dtype,
						'crs': crs,
						'transform': tile_transform,
					}
					
					# Add nodata value if present
					if nodata is not None:
						output_meta['nodata'] = nodata
					
					# Write the tile
					with rasterio.open(output_path, 'w', **output_meta) as dst:
						dst.write(data)
					
					output_files.append(output_path)
					print(f"Created: {output_filename}")
			
			print(f"\nTotal tiles created: {len(output_files)}")
			return output_files
			
	except rasterio.errors.RasterioIOError as e:
		raise IOError(f"Error reading GeoTIFF file: {e}")
	except Exception as e:
		raise RuntimeError(f"Error processing GeoTIFF: {e}")

site_name='LemonsAB'
geotiff_path = "./" + site_name + "/" + site_name + ".tif"
geotiff_split_output_dir = "./" + site_name + "/tiles_split/"

# Split the GeoTIFF into 256x256 pixel tiles
tiles = split_geotiff(
	input_path=geotiff_path,
	output_dir=geotiff_split_output_dir,
	tile_width=2048,
	tile_height=2048,
	skip_partial_tiles=False  # Set to True to skip edge tiles that don't fit evenly
)

print(f"\nOutput tiles saved to: {geotiff_split_output_dir}")